**Documentação do Notebook: Agrupamento1_HDBSCAN**

Este notebook tem como objetivo realizar o pré-processamento e a engenharia de features em três conjuntos de dados distintos: Telemetria, Histórico e Dados de Clientes (telemetria_n.csv, historico.csv e dados_clients.csv).

O processo consiste em carregar os dados, limpá-los, transformá-los e agregar as informações em um nível de cliente, preparando-os para serem utilizados em modelos de clusterização. Ao final de cada etapa de processamento, um arquivo Parquet consolidado é salvo.

Vale ressaltar que este notebook foi executado utilizando ***TPU v2-8 do Colab!*** Do contrário um ***ERRO*** de excesso de Memória RAM irá acontecer.

**Módulo 1: Configuração Inicial e Funções Auxiliares**

Esta seção é responsável por preparar o ambiente de análise de dados.

* Bibliotecas:

  1. pandas: Biblioteca fundamental para manipulação e análise de dados tabulares (DataFrames).

* Configurações de Exibição do Pandas:

  1. display.max_columns e display.max_rows: Configurados como None para garantir que, ao exibir um DataFrame, todas as suas colunas e linhas sejam mostradas, facilitando a inspeção visual completa dos dados.

  2. display.float_format: Formata todos os números de ponto flutuante para exibição com duas casas decimais, melhorando a legibilidade de valores monetários e métricas.

* Função de Leitura de CSV (try_read_csv):

  1. Define uma função robusta para carregar arquivos CSV. Ela tenta ler um arquivo sequencialmente com diferentes codificações (utf-8, latin1) e separadores (;), tratando exceções comuns de formatação. Isso torna o script mais resiliente a variações nos arquivos de origem. Se nenhuma combinação funcionar, a função levanta um erro claro.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

def try_read_csv(path):
    for encoding in ['utf-8', 'latin1']:
        for sep in [';']:
            try:
                return pd.read_csv(path, encoding=encoding, sep=sep)
            except Exception:
                continue
    raise ValueError(f'Erro ao ler o arquivo: {path}')

**Módulo 2: Carregamento dos Dados**

Nesta etapa, os arquivos CSV brutos são carregados na memória como DataFrames do pandas.

* Fontes de Dados: São carregados três conjuntos de dados principais:

  1. dados_clientes.csv: Informações cadastrais e contratuais dos clientes.

  2. historico.csv: Histórico de propostas comerciais.

  3. telemetria_*.csv: Onze arquivos separados contendo dados de telemetria de uso dos produtos.

* Estratégia de Carregamento:

  1. Os arquivos dados_clientes e historico são lidos usando a função auxiliar try_read_csv.

  2. Os 11 arquivos de telemetria são carregados individualmente com pd.read_csv, especificando a vírgula como separador, pois possuem um formato consistente. Essa abordagem é eficaz para gerenciar o uso de memória ao lidar com dados segmentados em múltiplos arquivos grandes.

In [ ]:
dados_clientes = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/dados_clientes.csv')
historico = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/historico.csv')
telemetria1 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_1.csv', sep=',')
telemetria2 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_2.csv', sep=',')
telemetria3 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_3.csv', sep=',')
telemetria4 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_4.csv', sep=',')
telemetria5 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_5.csv', sep=',')
telemetria6 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_6.csv', sep=',')
telemetria7 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_7.csv', sep=',')
telemetria8 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_8.csv', sep=',')
telemetria9 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_9.csv', sep=',')
telemetria10 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_10.csv', sep=',')
telemetria11 = pd.read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/telemetria_11.csv', sep=',')

**PRIMEIRA PARTE: Processamento dos Dados de Telemetria**

O objetivo desta etapa é consolidar, limpar e agregar os dados de uso dos produtos para criar um perfil de utilização para cada cliente.

* Consolidação e Engenharia de Features
  1. Concatenação: Os 11 DataFrames de telemetria são combinados em um único DataFrame (df_telemetria) usando pd.concat, com ignore_index=True para criar um novo índice contínuo.

  2. Renomeação e Padronização:

    * As colunas são renomeadas para um padrão mais claro e consistente (ex: clienteid para CD_CLIENTE).

    * O sufixo "00" é removido da coluna CD_CLIENTE para normalizar o identificador do cliente e garantir a compatibilidade com as outras tabelas.

  3. Engenharia de Features (Features de Tempo):

    * A coluna eventduration, originalmente em segundos, é usada para criar duas novas features:

      * EVENTDURATION_H: Duração do evento em horas.

      * EVENTDURATION_D: Duração do evento em dias.

    * Essas novas colunas facilitam a interpretação e análise da duração de uso em escalas de tempo mais humanas.

* Limpeza de Dados (Data Cleaning)

  1. Remoção de Dados Inválidos:

    * Registros com EVENTDURATION_S (duração do evento em segundos) negativos são removidos, pois são fisicamente impossíveis e possivelmente representam erros de dados, que podem ter ocorrido tanto na hora do preenchimento (em caso de processo manual) ou exportação.

    * Registros onde a duração do evento (EVENTDURATION_D) excede 15.330 dias (aproximadamente 42 anos) são filtrados. Esta regra de negócio foi baseada na idade da empresa parceira, assumindo que durações maiores que isso são erros de registro, mas com uma observação: No dicionário é apontado que moduloid é uma espécie de agrupador de produtos, logo, se EVENTDURATION_D é a soma da duração de todos os produtos neste módulo, o tempo PODE ser possível.

* Agregação por Cliente

  1. Os dados de telemetria, que estão no nível de evento, são agregados para criar uma visão resumida por cliente.

  2. Agregação da Duração do Evento:

    * A mediana da duração do evento em segundos (EVENTDURATION_S) é calculada para cada cliente. A mediana foi escolhida em vez da média por ser uma medida estatística mais robusta a valores extremos (outliers), que são comuns nestes dados de telemetria e poderiam distorcer a análise.

  3. Contagem de Status de Licença (Pivotagem):

    * Os dados são agrupados por cliente e status da licença (STATUS_LICENCA).

    * A função unstack() é utilizada para "pivotar" a contagem de status, transformando as categorias (Conectado, Desconectado, Negado) em colunas separadas. Isso cria features como LICENCA_Conectado, LICENCA_Desconectado, etc., que quantificam o comportamento de licenciamento de cada cliente.

  4. Merge e Exportação:

    * Os dados agregados de duração e contagem de licenças são unidos em um DataFrame final (df_telemetria_agrupado_final).

    * O resultado é salvo no formato Parquet, que é otimizado para armazenamento e performance em análises de big data e mantêm o tipo do dado.

In [ ]:
telemetrias = [telemetria1, telemetria2, telemetria3, telemetria4,
              telemetria5, telemetria6, telemetria7, telemetria8,
              telemetria9, telemetria10, telemetria11]

df_telemetria = pd.concat(telemetrias, ignore_index=True)

df_telemetria.rename(columns={'eventduration': 'eventduration_segundos'}, inplace=True)

df_telemetria['eventduration_horas'] = df_telemetria['eventduration_segundos'] / 3600
df_telemetria['eventduration_dias'] = df_telemetria['eventduration_segundos'] / (3600 * 24)

df_telemetria = df_telemetria.rename(columns={
    'clienteid': 'CD_CLIENTE',
    'eventduration_segundos': 'EVENTDURATION_S',
    'referencedatestart': 'DT_INICIO_EVENTO',
    'statuslicenca': 'STATUS_LICENCA',
    'eventduration_horas': 'EVENTDURATION_H',
    'eventduration_dias': 'EVENTDURATION_D'
})

df_telemetria.rename(columns={'CD_CLIENTE': 'CD_CLIENTE_ORIGINAL'}, inplace=True)

df_telemetria['CD_CLIENTE'] = df_telemetria['CD_CLIENTE_ORIGINAL'].str.removesuffix('00')

display(df_telemetria.head())

,CD_CLIENTE_ORIGINAL,EVENTDURATION_S,moduloid,productlineid,DT_INICIO_EVENTO,slotid,STATUS_LICENCA,tcloud,clienteprime,EVENTDURATION_H,EVENTDURATION_D,CD_CLIENTE
0,TEXKCV00,580.72,6,2,2025-03-19,4133,Desconectado,NaN,NaN,0.16,0.01,TEXKCV
1,TAAHTU00,414787.07,6,2,2025-03-19,4000,Desconectado,NaN,NaN,115.22,4.80,TAAHTU
2,TEXKCV00,4933.51,6,2,2025-03-19,4133,Desconectado,NaN,NaN,1.37,0.06,TEXKCV
3,TEWERU00,414086.47,7,2,2025-03-19,4000,Desconectado,NaN,NaN,115.02,4.79,TEWERU
4,TFDFHJ00,0.00,524,3,2025-03-19,0,Desconectado,NaN,NaN,0.00,0.00,TFDFHJ


In [ ]:
df_telemetria = df_telemetria[df_telemetria['EVENTDURATION_S'] >= 0]
df_telemetria = df_telemetria[df_telemetria['EVENTDURATION_D'] <= 15330]
display(df_telemetria.describe())

,EVENTDURATION_S,moduloid,productlineid,slotid,tcloud,clienteprime,EVENTDURATION_H,EVENTDURATION_D
count,30728265.00,30728265.00,30728265.00,30728265.00,0.00,0.00,30728265.00,30728265.00
mean,3650.86,1334.30,3.93,2025.15,NaN,NaN,1.01,0.04
std,152835.53,1910.81,2.79,1797.88,NaN,NaN,42.45,1.77
min,0.00,0.00,0.00,0.00,NaN,NaN,0.00,0.00
25%,11.24,516.00,3.00,533.00,NaN,NaN,0.00,0.00
50%,200.00,533.00,3.00,534.00,NaN,NaN,0.06,0.00
75%,1029.72,722.00,6.00,4001.00,NaN,NaN,0.29,0.01
max,261080800.00,7505.00,99.00,6500.00,NaN,NaN,72522.44,3021.77


In [ ]:
contagem_licenca = df_telemetria.groupby(['CD_CLIENTE', 'STATUS_LICENCA']).size().unstack(fill_value=0)
contagem_licenca.columns = ['LICENCA_' + col for col in contagem_licenca.columns]

df_telemetria_agrupado_final = df_telemetria.groupby('CD_CLIENTE').agg(
    EVENTDURATION_S=('EVENTDURATION_S', 'median'),
)

df_telemetria_agrupado_final = pd.merge(
    df_telemetria_agrupado_final,
    contagem_licenca,
    on='CD_CLIENTE',
    how='left'
)

In [ ]:
df_telemetria_agrupado_final.head()

,EVENTDURATION_S,LICENCA_Conectado,LICENCA_Desconectado,LICENCA_Negado
CD_CLIENTE,,,,
T00053,229.53,0.00,4400.00,0.00
T00082,301.73,0.00,3101.00,2.00
T00145,493.08,0.00,1016.00,4.00
T00245,91.35,0.00,279.00,24.00
T00255,0.00,0.00,1259.00,0.00


In [ ]:
df_telemetria_agrupado_final.reset_index(inplace=True)

In [ ]:
df_telemetria_agrupado_final.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/dados_telemetria_agrupado_final.parquet', index=False)

**SEGUNDA PARTE: Processamento do Histórico de Vendas**

Esta seção foca em tratar e agregar o histórico de propostas comerciais para extrair o perfil de compra de cada cliente.

* Preparação e Limpeza
  1. Conversão de Tipos de Dados:

    * Colunas numéricas que foram lidas como texto (devido ao uso de vírgula como separador decimal) são convertidas para o tipo float. Isso inclui valores monetários, quantidades e percentuais.

  2. Validação e Recálculo de Campos:

    * As colunas de valores (VL_DESCONTO, VL_TOTAL, etc.) são recalculadas a partir de valores base (PRC_UNITARIO, QTD, VL_PCT_DESCONTO) seguindo o que foi colocado pela própria empresa no dicionário de dados fornecido. Esta é uma etapa crucial para garantir a consistência e a integridade dos dados financeiros, em vez de confiarmos cegamente nos valores pré-calculados do dataset original, visto que já presenciamos dados distorcidos anteriormente com a telemetria.

* Agregação em Duas Etapas
  1. A agregação é feita em dois níveis para capturar tanto o perfil de uma proposta individual quanto o perfil geral do cliente.

    * Etapa 1: Agregação por Proposta (df_proposta_agrupado)

      * Os dados são primeiro agrupados por CD_CLIENTE e NR_PROPOSTA.

      * Para cada proposta, são calculadas métricas como:

        1. QTD_ITEMS_DISTINTOS_PROPOSTA: O número máximo de itens em uma proposta.

        2. QTD_UNIDADES_TOTAIS: A soma total de unidades de todos os itens.

        3. Medianas de descontos, preços e meses de bonificação.

    * Etapa 2: Agregação por Cliente (df_historico_agrupado_cliente)

      * Os resultados da agregação por proposta são então agrupados por CD_CLIENTE.

      * Isso gera o perfil final do cliente, com métricas como:

        1. QTD_PROPOSTAS_DISTINTAS: Quantidade de propostas que o cliente já teve.

        2. Medianas e somas das métricas calculadas na etapa anterior, resumindo o comportamento de compra do cliente ao longo do tempo.

        3. MEDIAN_UNIDADES_POR_PROPOSTA: Uma nova feature que calcula a quantidade média de unidades que um cliente costuma adquirir por proposta.

* Exportação
  1. O DataFrame final, com o perfil de compra de cada cliente, é salvo em formato Parquet.

In [ ]:
historico.head()

,NR_PROPOSTA,ITEM_PROPOSTA,DT_UPLOAD,HOSPEDAGEM,CD_CLI,FAT_FAIXA,CD_PROD,QTD,MESES_BONIF,VL_PCT_DESC_TEMP,VL_PCT_DESCONTO,PRC_UNITARIO,VL_DESCONTO_TEMPORARIO,VL_TOTAL,VL_FULL,VL_DESCONTO
0,AAMQSF,1,2025-03-25,ON PREMISES,TFDPFE,Sem Informações de Faturamento,0113301112,1,0,0,"28,6492879623732","2101,92868395988",0,"2101,92868395988","6599,31618873727","4497,38750477739"
1,AAJUVA,7,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000450,1,0,0,0,"0,53388988572581",0,"0,53388988572581","0,53388988572581",0
2,AAKX71,1,2024-08-21,ON PREMISES,T48463,Faixa 03 - De 15 M ate 25 M,1M13301050,1,0,0,0,"1222,89790447049",0,"1222,89790447049","63,0578605187964",0
3,AAMJNP,1,2025-02-17,ON PREMISES,TFEED1,Sem Informações de Faturamento,71A3301148,1,0,0,0,"60,1067526465167",0,"60,1067526465167","60,1067526465167",0
4,AAKFC4,1,2024-05-23,ON PREMISES,TDC1GA,Sem Informações de Faturamento,CONSV.502,"4,5",0,0,0,"93,9520083156387",0,"422,786139349058","0,0189173581556389",0


In [ ]:
historico.rename(columns={'CD_CLI': 'CD_CLIENTE'}, inplace=True)
historico['QTD'] = historico['QTD'].str.replace(',', '.').astype(float)
historico['MESES_BONIF'] = historico['MESES_BONIF'].astype(float)
historico['VL_PCT_DESC_TEMP'] = historico['VL_PCT_DESC_TEMP'].str.replace(',', '.').astype(float)
historico['VL_PCT_DESCONTO'] = historico['VL_PCT_DESCONTO'].str.replace(',', '.').astype(float)
historico['PRC_UNITARIO'] = historico['PRC_UNITARIO'].str.replace(',', '.').astype(float)
historico['VL_DESCONTO_TEMPORARIO'] = historico['VL_DESCONTO_TEMPORARIO'].str.replace(',', '.').astype(float)
historico['VL_TOTAL'] = historico['VL_TOTAL'].str.replace(',', '.').astype(float)
historico['VL_FULL'] = historico['VL_FULL'].str.replace(',', '.').astype(float)
historico['VL_DESCONTO'] = historico['VL_DESCONTO'].str.replace(',', '.').astype(float)

In [ ]:
historico['VL_DESCONTO'] = historico['PRC_UNITARIO'] * historico['QTD'] * (historico['VL_PCT_DESCONTO']/100)

In [ ]:
historico['PRC_UNITARIO_TAB'] = historico['PRC_UNITARIO'] + historico['VL_DESCONTO']

In [ ]:
historico['VL_DESCONTO_TEMPORARIO'] = historico['PRC_UNITARIO'] * historico['QTD'] * (historico['VL_PCT_DESC_TEMP']/100)

In [ ]:
historico['VL_TOTAL'] = historico['PRC_UNITARIO'] * historico['QTD']

In [ ]:
historico['VL_FULL'] = historico['PRC_UNITARIO_TAB'] * historico['QTD']

In [ ]:
historico.head()

,NR_PROPOSTA,ITEM_PROPOSTA,DT_UPLOAD,HOSPEDAGEM,CD_CLIENTE,FAT_FAIXA,CD_PROD,QTD,MESES_BONIF,VL_PCT_DESC_TEMP,VL_PCT_DESCONTO,PRC_UNITARIO,VL_DESCONTO_TEMPORARIO,VL_TOTAL,VL_FULL,VL_DESCONTO,PRC_UNITARIO_TAB
0,AAMQSF,1,2025-03-25,ON PREMISES,TFDPFE,Sem Informações de Faturamento,0113301112,1.00,0.00,0.00,28.65,2101.93,0.00,2101.93,2704.12,602.19,2704.12
1,AAJUVA,7,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000450,1.00,0.00,0.00,0.00,0.53,0.00,0.53,0.53,0.00,0.53
2,AAKX71,1,2024-08-21,ON PREMISES,T48463,Faixa 03 - De 15 M ate 25 M,1M13301050,1.00,0.00,0.00,0.00,1222.90,0.00,1222.90,1222.90,0.00,1222.90
3,AAMJNP,1,2025-02-17,ON PREMISES,TFEED1,Sem Informações de Faturamento,71A3301148,1.00,0.00,0.00,0.00,60.11,0.00,60.11,60.11,0.00,60.11
4,AAKFC4,1,2024-05-23,ON PREMISES,TDC1GA,Sem Informações de Faturamento,CONSV.502,4.50,0.00,0.00,0.00,93.95,0.00,422.78,422.78,0.00,93.95


Checagem para entender a disposição, ITEM_PROPOSTA, na verdade é incremental, é errôneo somar, por isso tanto problema antes. O restante dos valores é individual, aparentemente.

In [ ]:
aalvej_proposals = historico[historico['NR_PROPOSTA'] == 'AAJUVA']
display(aalvej_proposals.sort_values(by='ITEM_PROPOSTA', ascending=False))

,NR_PROPOSTA,ITEM_PROPOSTA,DT_UPLOAD,HOSPEDAGEM,CD_CLIENTE,FAT_FAIXA,CD_PROD,QTD,MESES_BONIF,VL_PCT_DESC_TEMP,VL_PCT_DESCONTO,PRC_UNITARIO,VL_DESCONTO_TEMPORARIO,VL_TOTAL,VL_FULL,VL_DESCONTO,PRC_UNITARIO_TAB
21083,AAJUVA,12,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000557,1.00,0.00,0.00,0.00,306.10,0.00,306.10,306.10,0.00,306.10
13737,AAJUVA,11,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000566,1.00,0.00,0.00,0.00,306.10,0.00,306.10,306.10,0.00,306.10
21339,AAJUVA,10,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000731,1.00,0.00,0.00,0.00,163.13,0.00,163.13,163.13,0.00,163.13
3884,AAJUVA,9,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000729,1.00,0.00,0.00,0.00,163.13,0.00,163.13,163.13,0.00,163.13
22335,AAJUVA,8,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000728,1.00,0.00,0.00,0.00,163.13,0.00,163.13,163.13,0.00,163.13
1,AAJUVA,7,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000450,1.00,0.00,0.00,0.00,0.53,0.00,0.53,0.53,0.00,0.53
66,AAJUVA,6,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000443,1.00,0.00,0.00,0.00,506.41,0.00,506.41,506.41,0.00,506.41
7518,AAJUVA,5,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000565,1.00,0.00,0.00,0.00,0.60,0.00,0.60,0.60,0.00,0.60
12380,AAJUVA,4,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000558,1.00,0.00,0.00,0.00,353.19,0.00,353.19,353.19,0.00,353.19
18977,AAJUVA,3,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000574,1.00,0.00,0.00,0.00,0.60,0.00,0.60,0.60,0.00,0.60


In [ ]:
df_proposta_agrupado = historico.groupby(['CD_CLIENTE', 'NR_PROPOSTA']).agg(
    QTD_ITEMS_DISTINTOS_PROPOSTA=('ITEM_PROPOSTA', 'max'),
    QTD_UNIDADES_TOTAIS=('QTD', 'sum'),
    MEDIAN_MESES_BONIF=('MESES_BONIF', 'median'),
    MEDIAN_VL_PCT_DESCONTO=('VL_PCT_DESCONTO', 'median'),
    MEDIAN_VL_PCT_DESC_TEMP=('VL_PCT_DESC_TEMP', 'median'),
    MEDIAN_PRC_UNITARIO_POR_PROPOSTA=('PRC_UNITARIO', 'median'),
    VALOR_TOTAL_PROPOSTA=('PRC_UNITARIO', 'sum'),
).reset_index()

In [ ]:
df_proposta_agrupado.head()

,CD_CLIENTE,NR_PROPOSTA,QTD_ITEMS_DISTINTOS_PROPOSTA,QTD_UNIDADES_TOTAIS,MEDIAN_MESES_BONIF,MEDIAN_VL_PCT_DESCONTO,MEDIAN_VL_PCT_DESC_TEMP,MEDIAN_PRC_UNITARIO_POR_PROPOSTA,VALOR_TOTAL_PROPOSTA
0,T00053,AAKHCJ,2,51.00,0.00,0.00,0.00,686.36,1372.72
1,T00082,AAKGE4,3,3.00,499.50,0.00,11.01,1381.88,2763.77
2,T00082,AAKNZV,2,5.00,999.00,0.00,17.94,73.32,73.32
3,T00082,AAMFMQ,1,26.50,0.00,0.00,0.00,58.58,58.58
4,T00082,AAMG78,1,1.00,0.00,0.00,0.00,351.81,351.81


In [ ]:
historico.head()

,NR_PROPOSTA,ITEM_PROPOSTA,DT_UPLOAD,HOSPEDAGEM,CD_CLIENTE,FAT_FAIXA,CD_PROD,QTD,MESES_BONIF,VL_PCT_DESC_TEMP,VL_PCT_DESCONTO,PRC_UNITARIO,VL_DESCONTO_TEMPORARIO,VL_TOTAL,VL_FULL,VL_DESCONTO,PRC_UNITARIO_TAB
0,AAMQSF,1,2025-03-25,ON PREMISES,TFDPFE,Sem Informações de Faturamento,0113301112,1.00,0.00,0.00,28.65,2101.93,0.00,2101.93,2704.12,602.19,2704.12
1,AAJUVA,7,2024-03-28,ON PREMISES,T03306,Faixa 08 - De 150 M ate 300 M,AUT.04.000450,1.00,0.00,0.00,0.00,0.53,0.00,0.53,0.53,0.00,0.53
2,AAKX71,1,2024-08-21,ON PREMISES,T48463,Faixa 03 - De 15 M ate 25 M,1M13301050,1.00,0.00,0.00,0.00,1222.90,0.00,1222.90,1222.90,0.00,1222.90
3,AAMJNP,1,2025-02-17,ON PREMISES,TFEED1,Sem Informações de Faturamento,71A3301148,1.00,0.00,0.00,0.00,60.11,0.00,60.11,60.11,0.00,60.11
4,AAKFC4,1,2024-05-23,ON PREMISES,TDC1GA,Sem Informações de Faturamento,CONSV.502,4.50,0.00,0.00,0.00,93.95,0.00,422.78,422.78,0.00,93.95


In [ ]:
historico_unique = historico[['CD_CLIENTE', 'FAT_FAIXA', 'NR_PROPOSTA']]

In [ ]:
historico_unique.head()

,CD_CLIENTE,FAT_FAIXA,NR_PROPOSTA
0,TFDPFE,Sem Informações de Faturamento,AAMQSF
1,T03306,Faixa 08 - De 150 M ate 300 M,AAJUVA
2,T48463,Faixa 03 - De 15 M ate 25 M,AAKX71
3,TFEED1,Sem Informações de Faturamento,AAMJNP
4,TDC1GA,Sem Informações de Faturamento,AAKFC4


In [ ]:
historico_merged = historico_unique.merge(df_proposta_agrupado, on=['CD_CLIENTE', 'NR_PROPOSTA'], how='left')

In [ ]:
historico_merged.head()

,CD_CLIENTE,FAT_FAIXA,NR_PROPOSTA,QTD_ITEMS_DISTINTOS_PROPOSTA,QTD_UNIDADES_TOTAIS,MEDIAN_MESES_BONIF,MEDIAN_VL_PCT_DESCONTO,MEDIAN_VL_PCT_DESC_TEMP,MEDIAN_PRC_UNITARIO_POR_PROPOSTA,VALOR_TOTAL_PROPOSTA
0,TFDPFE,Sem Informações de Faturamento,AAMQSF,1,2.00,0.00,14.46,0.00,1219.12,2438.24
1,T03306,Faixa 08 - De 150 M ate 300 M,AAJUVA,12,12.00,0.00,0.00,0.00,234.62,14459.49
2,T48463,Faixa 03 - De 15 M ate 25 M,AAKX71,1,1.00,0.00,0.00,0.00,1222.90,1222.90
3,TFEED1,Sem Informações de Faturamento,AAMJNP,1,1.00,0.00,0.00,0.00,60.11,60.11
4,TDC1GA,Sem Informações de Faturamento,AAKFC4,1,4.50,0.00,0.00,0.00,93.95,93.95


In [ ]:
df_historico_agrupado_cliente = historico_merged.groupby('CD_CLIENTE').agg(
    QTD_PROPOSTAS_DISTINTAS=('NR_PROPOSTA', 'nunique'),
    QTD_ITEMS_DISTINTOS_PROPOSTA=('QTD_ITEMS_DISTINTOS_PROPOSTA', 'median'),
    QTD_UNIDADES_TOTAIS=('QTD_UNIDADES_TOTAIS', 'sum'),
    MEDIAN_MESES_BONIF=('MEDIAN_MESES_BONIF', 'median'),
    MEDIAN_VL_PCT_DESCONTO=('MEDIAN_VL_PCT_DESCONTO', 'median'),
    MEDIAN_VL_PCT_DESC_TEMP=('MEDIAN_VL_PCT_DESC_TEMP', 'median'),
    MEDIAN_PRC_UNITARIO_POR_PROPOSTA=('MEDIAN_PRC_UNITARIO_POR_PROPOSTA', 'median'),
    MEDIAN_VALOR_TOTAL_PROPOSTA=('VALOR_TOTAL_PROPOSTA', 'median'),
).reset_index()

In [ ]:
df_historico_agrupado_cliente['MEDIAN_UNIDADES_POR_PROPOSTA'] = df_historico_agrupado_cliente['QTD_UNIDADES_TOTAIS'] / df_historico_agrupado_cliente['QTD_PROPOSTAS_DISTINTAS']

In [ ]:
df_historico_agrupado_cliente.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/df_historico_agrupado_cliente.parquet', index=False)

In [ ]:
df_historico_agrupado_cliente.head()

,CD_CLIENTE,QTD_PROPOSTAS_DISTINTAS,QTD_ITEMS_DISTINTOS_PROPOSTA,QTD_UNIDADES_TOTAIS,MEDIAN_MESES_BONIF,MEDIAN_VL_PCT_DESCONTO,MEDIAN_VL_PCT_DESC_TEMP,MEDIAN_PRC_UNITARIO_POR_PROPOSTA,MEDIAN_VALOR_TOTAL_PROPOSTA,MEDIAN_UNIDADES_POR_PROPOSTA
0,T00053,1,2.00,102.00,0.00,0.00,0.00,686.36,1372.72,102.00
1,T00082,6,1.00,77.50,0.00,0.00,0.00,194.45,194.45,12.92
2,T00145,1,1.00,10.00,0.00,0.00,0.00,2.35,2.35,10.00
3,T00336,2,1.00,2.00,0.00,0.00,0.00,8611.75,8611.75,1.00
4,T00673,1,2.00,16.00,0.00,13.45,0.00,240.29,6782.37,16.00


**TERCEIRA PARTE: Processamento dos Dados de Clientes**

Esta etapa final trata os dados cadastrais e contratuais, assumimos que esta é a tabela principal que unirá todas as informações.

* Tratamento Inicial

  1. Conversão de Tipos: A coluna VL_TOTAL_CONTRATO é convertida para float, e DT_ASSINATURA_CONTRATO para o formato datetime.

  2. Remoção de Colunas: As colunas PAIS e CIDADE são removidas, pois consideramos utilizar apenas UF como variável geográfica.

* Agregação e Engenharia de Features

  1. Pivotagem de Dados Contratuais:
    * Assim como na seção de telemetria, a técnica de groupby().size().unstack() é usada para transformar dados categóricos em features numéricas:

      * Situação dos Contratos: Cria colunas como CONTRATOS_ATIVO, CONTRATOS_CANCELADO, etc., contando quantos contratos cada cliente tem em cada situação.

      * Periodicidade de Pagamento: Cria colunas como PERIODICIDADE_MENSAL, PERIODICIDADE_ANUAL, etc.

    * De forma similar, groupby().sum().unstack() é usado para calcular o valor total por situação de contrato (ex: VALOR_CONTRATOS_ATIVO).

  2. Agregação de Informações Gerais:

    * Um groupby final em CD_CLIENTE calcula métricas-chave como:

      1. QTD_PRODUTOS_DISTINTOS: Número de produtos únicos que o cliente possui.

      2. SOMA_VL_CONTRATOS: Valor total somado de todos os contratos do cliente.

      3. LISTA_PRODUTOS: Uma lista com todos os produtos em contrato ou já contratados.

* Consolidação Final
  1. Merge dos Dados: Os vários DataFrames agregados (contagens, valores, etc.) são unidos (pd.merge) em uma única tabela.

  2. Adição de Dados Cadastrais: As informações estáticas do cliente (como DS_SEGMENTO, UF, FAT_FAIXA) são adicionadas ao DataFrame final. drop_duplicates() é usado para garantir que cada cliente apareça apenas uma vez.

* Exportação
  1. O DataFrame final e enriquecido dos clientes é salvo em formato Parquet, pronto para ser combinado com os outros datasets para análise ou modelagem.

In [ ]:
dados_clientes.head()

,CD_CLIENTE,DS_PROD,DS_LIN_REC,CIDADE,DS_CNAE,DS_SEGMENTO,DS_SUBSEGMENTO,FAT_FAIXA,MARCA_TOTVS,MODAL_COMERC,PAIS,PERIODICIDADE,SITUACAO_CONTRATO,UF,VL_TOTAL_CONTRATO,DT_ASSINATURA_CONTRATO
0,99958,SMS FULL TOTVS TRAD,SMS TOTVS SERIE T,JOINVILLE,PESSOA FISICA (SEM CNAE),SERVICOS,PROVEDOR SERVICOS,Faixa 09 - De 300 M ate 500 M,CROSS - TRADICIONAL,MODALIDADE TRADICIONAL,105,00 - Mensal,GRATUITO,SC,"1633817,36581438",2016-04-07
1,T00053,SMS COLAB NEO 2500 DOC,SMS TOTVS SERIE T,RIODEJANEIRO,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,MANUFATURA - PARCEIRO,MODALIDADE TRADICIONAL,105,00 - Mensal,ATIVO,RJ,"341,155636978792",2015-02-27
2,T00053,HORA SUPORTE,CONSULTORIA TRADICIONAL,RIODEJANEIRO,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,SERVICOS DE IMPLANTACAO,MODALIDADE SERVICOS NÃO RECORRENTES,105,00 - Mensal,CANCELADO,RJ,"45,3386017130146",1997-11-28
3,99958,CDU FULL TOTVS TRAD,CDU TOTVS SERIE T,JOINVILLE,PESSOA FISICA (SEM CNAE),SERVICOS,PROVEDOR SERVICOS,Faixa 09 - De 300 M ate 500 M,CROSS - TRADICIONAL,MODALIDADE TRADICIONAL,105,00 - Mensal,GRATUITO,SC,"42,0343698218297",2016-04-07
4,T00053,PROGRESS USER 11 CDU,CDU TOTVS SERIE T,RIODEJANEIRO,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,PROGRESS,MODALIDADE TRADICIONAL,105,00 - Mensal,TROCADO,RJ,"0,117708006301753",2017-11-22


In [ ]:
dados_clientes['VL_TOTAL_CONTRATO'] = dados_clientes['VL_TOTAL_CONTRATO'].str.replace(',', '.').astype(float)
dados_clientes['DT_ASSINATURA_CONTRATO'] = pd.to_datetime(dados_clientes['DT_ASSINATURA_CONTRATO'])

In [ ]:
dados_clientes.drop(columns=['PAIS', 'CIDADE'], inplace=True)

In [ ]:
dados_clientes.head()

,CD_CLIENTE,DS_PROD,DS_LIN_REC,DS_CNAE,DS_SEGMENTO,DS_SUBSEGMENTO,FAT_FAIXA,MARCA_TOTVS,MODAL_COMERC,PERIODICIDADE,SITUACAO_CONTRATO,UF,VL_TOTAL_CONTRATO,DT_ASSINATURA_CONTRATO
0,99958,SMS FULL TOTVS TRAD,SMS TOTVS SERIE T,PESSOA FISICA (SEM CNAE),SERVICOS,PROVEDOR SERVICOS,Faixa 09 - De 300 M ate 500 M,CROSS - TRADICIONAL,MODALIDADE TRADICIONAL,00 - Mensal,GRATUITO,SC,1633817.37,2016-04-07
1,T00053,SMS COLAB NEO 2500 DOC,SMS TOTVS SERIE T,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,MANUFATURA - PARCEIRO,MODALIDADE TRADICIONAL,00 - Mensal,ATIVO,RJ,341.16,2015-02-27
2,T00053,HORA SUPORTE,CONSULTORIA TRADICIONAL,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,SERVICOS DE IMPLANTACAO,MODALIDADE SERVICOS NÃO RECORRENTES,00 - Mensal,CANCELADO,RJ,45.34,1997-11-28
3,99958,CDU FULL TOTVS TRAD,CDU TOTVS SERIE T,PESSOA FISICA (SEM CNAE),SERVICOS,PROVEDOR SERVICOS,Faixa 09 - De 300 M ate 500 M,CROSS - TRADICIONAL,MODALIDADE TRADICIONAL,00 - Mensal,GRATUITO,SC,42.03,2016-04-07
4,T00053,PROGRESS USER 11 CDU,CDU TOTVS SERIE T,Fabricacao de preparacoes farmaceuticas,MANUFATURA,BENS DURAVEIS,Faixa 05 - De 35 M ate 50 M,PROGRESS,MODALIDADE TRADICIONAL,00 - Mensal,TROCADO,RJ,0.12,2017-11-22


In [ ]:
contagem_situacoes = dados_clientes.groupby(['CD_CLIENTE', 'SITUACAO_CONTRATO']).size().unstack(fill_value=0)
contagem_situacoes.columns = ['CONTRATOS_' + col for col in contagem_situacoes.columns]

In [ ]:
valor_situacoes = dados_clientes.groupby(['CD_CLIENTE', 'SITUACAO_CONTRATO'])['VL_TOTAL_CONTRATO'].sum().unstack(fill_value=0)
valor_situacoes.columns = ['VALOR_CONTRATOS_' + col for col in valor_situacoes.columns]

In [ ]:
valor_periodicidades = dados_clientes.groupby(['CD_CLIENTE', 'PERIODICIDADE']).size().unstack(fill_value=0)
valor_periodicidades.columns = ['PERIODICIDADE_' + col for col in valor_periodicidades.columns]

In [ ]:
df_dados_clientes_agregado = dados_clientes.groupby('CD_CLIENTE').agg(
    QTD_PRODUTOS_DISTINTOS=('DS_PROD', 'nunique'),
    QTD_TOTAL_PRODUTOS=('DS_PROD', 'count'),
    SOMA_VL_CONTRATOS=('VL_TOTAL_CONTRATO', 'sum'),
    LISTA_PRODUTOS=('DS_PROD', lambda x: list(x))
)

df_dados_clientes_agregado = pd.merge(
    df_dados_clientes_agregado,
    contagem_situacoes,
    on='CD_CLIENTE',
    how='left'
)

df_dados_clientes_agregado = pd.merge(
    df_dados_clientes_agregado,
    valor_situacoes,
    on='CD_CLIENTE',
    how='left'
)

df_dados_clientes_agregado = pd.merge(
    df_dados_clientes_agregado,
    valor_periodicidades,
    on='CD_CLIENTE',
    how='left'
)

df_dados_clientes_final = pd.merge(
    df_dados_clientes_agregado,
    dados_clientes[['CD_CLIENTE', 'DS_SEGMENTO', 'DS_SUBSEGMENTO', 'FAT_FAIXA', 'UF']].drop_duplicates(),
    on='CD_CLIENTE',
    how='left'
)

In [ ]:
df_dados_clientes_final.head()

,CD_CLIENTE,QTD_PRODUTOS_DISTINTOS,QTD_TOTAL_PRODUTOS,SOMA_VL_CONTRATOS,LISTA_PRODUTOS,CONTRATOS_ATIVO,CONTRATOS_CANCELADO,CONTRATOS_FATURAR,CONTRATOS_GRATUITO,CONTRATOS_PENDENTE,CONTRATOS_SUSPENSO,CONTRATOS_TROCADO,VALOR_CONTRATOS_ATIVO,VALOR_CONTRATOS_CANCELADO,VALOR_CONTRATOS_FATURAR,VALOR_CONTRATOS_GRATUITO,VALOR_CONTRATOS_PENDENTE,VALOR_CONTRATOS_SUSPENSO,VALOR_CONTRATOS_TROCADO,PERIODICIDADE_00 - Mensal,PERIODICIDADE_01 - Bimestral,PERIODICIDADE_02 - Trimestral,PERIODICIDADE_03 - Quadrimestral,PERIODICIDADE_05 - Semestral,PERIODICIDADE_11 - Anual,DS_SEGMENTO,DS_SUBSEGMENTO,FAT_FAIXA,UF
0,99069,6,6,8120.77,"[TOTVS RH ATS PACK 3 VAGAS, CLOUD IAAS 36M, SE...",2,2,0,2,0,0,0,651.60,7469.17,0.00,0.01,0.00,0.00,0.00,6,0,0,0,0,0,VAREJO,VAREJO,Sem Informações de Faturamento,SP
1,99958,8,8,1634142.18,"[SMS FULL TOTVS TRAD, CDU FULL TOTVS TRAD, SMS...",0,0,0,8,0,0,0,0.00,0.00,0.00,1634142.18,0.00,0.00,0.00,8,0,0,0,0,0,SERVICOS,PROVEDOR SERVICOS,Faixa 09 - De 300 M ate 500 M,SC
2,99999,334,356,42.72,"[AG.WN01.Mnt, HORA SUPORTE, BL.WN01.Mnt, BN.WN...",0,354,0,2,0,0,0,0.00,42.71,0.00,0.01,0.00,0.00,0.00,356,0,0,0,0,0,TOTVS,TOTVS,Sem Informações de Faturamento,SP
3,CARAMU,2,2,85314.57,"[ADESAO OT LOG PLA 500 VIAGENS, TOTVS OT LOG P...",2,0,0,0,0,0,0,85314.57,0.00,0.00,0.00,0.00,0.00,0.00,2,0,0,0,0,0,MANUFATURA,BENS DE CONSUMO,Sem Informações de Faturamento,GO
4,T00018,8,10,205.12,"[FEE - GDS INTERNACIONAL, ASSINATURA MENSAL IN...",1,4,0,5,0,0,0,203.06,2.00,0.00,0.06,0.00,0.00,0.00,10,0,0,0,0,0,SERVICOS,VIAGENS,"Faixa 02 - De 7,5 M ate 15 M",SP


In [ ]:
df_dados_clientes_final.to_parquet(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/dados_clientes_final.parquet', index=False)